# 🚰 Notebook 2: Leaky Bucket

**Leaky bucket** = a queue that drains at a fixed rate. Requests can pile up (up to a cap) but are *served* at a steady tempo. Useful when a downstream needs **smooth** traffic (no bursts at all).


## 🛠️ Setup

```bash
cd 04-patterns/rate-limiting-and-throttling
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟩 Implementation

In [ ]:
import time

class LeakyBucket:
    def __init__(self, leak_rate, capacity):
        self.leak_rate = leak_rate  # requests served per second
        self.capacity = capacity    # max queued
        self.queued = 0.0
        self.last = time.monotonic()

    def allow(self):
        now = time.monotonic()
        # Leak whatever should have drained since last check
        self.queued = max(0.0, self.queued - (now - self.last) * self.leak_rate)
        self.last = now
        if self.queued + 1 <= self.capacity:
            self.queued += 1
            return True
        return False


In [ ]:
lb = LeakyBucket(leak_rate=5, capacity=10)
burst = [int(lb.allow()) for _ in range(20)]
print('burst result:', burst, '— allowed', sum(burst))


## 🧠 Token vs Leaky

| | Token bucket | Leaky bucket |
|---|---|---|
| Allows bursts? | yes (up to bucket size) | no — output is paced |
| Rejects when? | bucket empty | queue full |
| Output rate | up to peak burst | constant `leak_rate` |
| Best for | API quotas, user-friendly | shaping traffic to a fragile downstream |
